# E3 -- extended Muller-Brown (10D): run notebook

**This notebook runs and saves. It does not typeset figures.**

Every official metric is computed here, at run time, and written into each run's `metrics_timeseries.csv` and `cost_timeseries.csv`. The companion notebook `E3_muller_brown_plot.ipynb` reads those numbers and never recomputes them.

**Run All executes the single default full configuration.** There is exactly one configuration for this experiment, `configs/experiments/E3.yaml` -- there is no smoke, dev, reduced, or production profile to choose between. Lowering the particle count for local debugging is an explicit temporary edit, never a second committed profile.

Each variant is saved the moment it finishes, into its own atomically renamed run directory, so a variant that fails leaves the earlier ones untouched.

In [1]:
import sys

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.pipeline import load_experiment, run_variants_and_save

## Target, reference, and cost calibration

The reference is built **once** and reused by every method. It does not depend on any method parameter, so it is **never rebuilt per method, per hyperparameter value, or per canonical/tamed variant**; a cached reference on disk is loaded instead of being recomputed.

The force-equivalent-evaluation (FEE) calibration is measured once per device in the same way, and every run in this experiment is costed against that one calibration. The device is resolved automatically -- no device index is pinned in this notebook.

In [2]:
experiment = load_experiment("E3", device="auto")

reference = experiment.ensure_reference()
fee = experiment.ensure_fee_calibration()

described = reference.describe()
print(f"reference: kind={described.get('kind', described.get('method'))}  hash={experiment.reference_hash}")
print(f"FEE:       unit={fee.cost_unit}  hash={fee.hash}")

reference: kind=cv_grid_density  hash=53af2ac3a1c61f75142147aab9cae27b
FEE:       unit=amortized_time_per_configuration  hash=088ecf363630e7e9c4899a3eb827d2f6


## ULA

Every taming-capable method runs **both** a canonical and a tamed variant. `run_variants_and_save` expands each entry of `variants` into those two runs by itself, so a notebook never passes `tame`. The two variants are **calibrated separately** -- each one gets its own step size from its own `dt` refinement -- and each is saved as its own run directory.

In [3]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULA",
    # `run_variants_and_save` expands each entry below into a
    # canonical and a tamed run, so `tame` is never passed here.
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[ULA, canonical] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/ULA/ULA-canonical-dt0.005-20260806T214833889605Z


[ULA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/ULA/ULA-tamed-dt0.005-20260806T214848405031Z


[{'variant_label': 'ULA, canonical',
  'status': 'complete',
  'run_id': 'ULA-canonical-dt0.005-20260806T214833889605Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/ULA/ULA-canonical-dt0.005-20260806T214833889605Z',
  'dt': 0.005,
  'calibration_hash': '96ed516683b3b77f3357a439392de12a',
  'fee_calibration_hash': '088ecf363630e7e9c4899a3eb827d2f6',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'ULA, tamed',
  'status': 'complete',
  'run_id': 'ULA-tamed-dt0.005-20260806T214848405031Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/ULA/ULA-tamed-dt0.005-20260806T214848405031Z',
  'dt': 0.005,
  'calibration_hash': 'bb66213e9e71b40940e28ae343f03a4d',
  'fee_calibration_hash': '088ecf363630e7e9c4899a3eb827d2f6',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## MALA

MALA supports taming, so it also runs both variants. Tamed MALA implements the actual tamed proposal density in the Metropolis-Hastings ratio; it is a genuine second sampler, not a relabelled copy of the canonical run.

In [4]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="MALA",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[MALA, canonical] NOT CALIBRATABLE: unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks


[MALA, tamed] NOT CALIBRATABLE: unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks


[{'variant_label': 'MALA, canonical',
  'method': 'MALA',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.04,
    'pass': False,
    'stability_problems': [('temporal_ess_fraction', 0.020262931787192944)],
    'agreement_failures': [],
    'summary': {'n_steps': 312,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.0,
     'temporal_ess': 6.322034717604199,
     'temporal_ess_fraction': 0.020262931787192944,
     'temporal_ess_draws_per_seed': 156,
     'n_effective': 1024,
     'summary_mean': 0.36726997862181676,
     'summary_mean_se': 0.010135231061045531,
     'summary_abs_mean': 0.41885374517913687,
     'summary_abs_mean_se': 0.007438273521610192,
     'summary_median': 0.49370867781672834,
     'summary_median_se': 0.008535731457758128,
     'summary_iqr': 0.499

## FLA

The three stability indices are this experiment's default grid in `configs/registry.yaml`. All three run from this one cell and save as separate variants, and each of them is expanded into a canonical and a tamed run.

In [5]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="FLA",
    variants=[
        {"alpha": 1.6}, {"alpha": 1.7}, {"alpha": 1.8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[FLA alpha=1.6, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) though it improves as the timestep shrinks


[FLA alpha=1.6, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/FLA/FLA-alpha1.6-tamed-dt0.0025-20260806T215253268643Z


[FLA alpha=1.7, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) though it improves as the timestep shrinks


[FLA alpha=1.7, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/FLA/FLA-alpha1.7-tamed-dt0.0025-20260806T215447092722Z


[FLA alpha=1.8, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) though it improves as the timestep shrinks


[FLA alpha=1.8, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/FLA/FLA-alpha1.8-tamed-dt0.0025-20260806T215641153016Z


[{'variant_label': 'FLA alpha=1.6, canonical',
  'method': 'FLA',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) though it improves as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.005,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.4291703125)],
    'agreement_failures': [{'key': 'energy_median',
      'coarse': 13.6418307,
      'fine': -0.17123367,
      'difference': 13.81306436,
      'allowance': 4.02777926}],
    'summary': {'n_steps': 2500,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.4291703125,
     'n_effective': 1024,
     'summary_mean': 0.852320439692279,
     'summary_mean_se': 0.02502773523668465,
     'summary_abs_mean': 1.0276665465718384,
     'summary_abs_mean_se': 0.01847031985904756,
     'summary_median': 1.111251450804339,
     'summary_median_se': 0.03487161617283635,
     'summary_iqr': 1.4007172953114908,
 

## ULD

ULD is the method; BAOAB is the integrator it is discretised with. Runs, manifests, and legends say ULD.

In [6]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULD",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[ULD gamma=1, canonical] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/ULD/ULD-gamma1-canonical-dt0.005-20260806T215655249681Z


[ULD gamma=1, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/ULD/ULD-gamma1-tamed-dt0.005-20260806T215710324328Z


[{'variant_label': 'ULD gamma=1, canonical',
  'status': 'complete',
  'run_id': 'ULD-gamma1-canonical-dt0.005-20260806T215655249681Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/ULD/ULD-gamma1-canonical-dt0.005-20260806T215655249681Z',
  'dt': 0.005,
  'calibration_hash': '8baddc73475d70b75f5b797957df8873',
  'fee_calibration_hash': '088ecf363630e7e9c4899a3eb827d2f6',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'ULD gamma=1, tamed',
  'status': 'complete',
  'run_id': 'ULD-gamma1-tamed-dt0.005-20260806T215710324328Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/ULD/ULD-gamma1-tamed-dt0.005-20260806T215710324328Z',
  'dt': 0.005,
  'calibration_hash': '14e1efdf8beb5f631efd666314172dfd',
  'fee_calibration_hash': '088ecf363630e7e9c4899a3eb827d2f6',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## PT

Parallel tempering. The replica ladder is tuned by the calibration step that `run_variants_and_save` invokes, not here, and the tuned ladder is written into the run's `calibration.json`.

In [7]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="PT",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[PT n_swap=10, canonical] NOT CALIBRATABLE: no timestep agreed with its halving on summary_mean, summary_median


[PT n_swap=10, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/PT/PT-n_swap10-tamed-dt0.04-20260806T220006551145Z


[{'variant_label': 'PT n_swap=10, canonical',
  'method': 'PT',
  'status': 'uncalibratable',
  'diagnosis': 'no timestep agreed with its halving on summary_mean, summary_median',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.08,
    'pass': False,
    'stability_problems': [],
    'agreement_failures': [{'key': 'summary_mean',
      'coarse': 0.22359269,
      'fine': 0.10833664,
      'difference': 0.11525605,
      'allowance': 0.08465207},
     {'key': 'summary_median',
      'coarse': 0.32065402,
      'fine': 0.03658652,
      'difference': 0.2840675,
      'allowance': 0.19533821}],
    'summary': {'n_steps': 156,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.00020188551682692306,
     'n_effective': 1024,
     'summary_mean': 0.22359269184161357,
     'summary_mean_se': 0.0120183543837823,
     'summary_abs_mean': 0.35727033217466586,
     'summary_abs_mean_se': 0.00759094677724247,
     'summary_median': 0.32065402256536135,
     'summ

## Raw-CP

The same compound-Poisson jump process with the Levy score correction switched off. It does not preserve the target, so it is the control arm that isolates what the score correction buys, not a competitive baseline.

In [8]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="Raw-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[Raw-CP, canonical] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/Raw-CP/Raw-CP-canonical-dt0.005-20260806T220027579532Z


[Raw-CP, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/Raw-CP/Raw-CP-tamed-dt0.005-20260806T220049686722Z


[{'variant_label': 'Raw-CP, canonical',
  'status': 'complete',
  'run_id': 'Raw-CP-canonical-dt0.005-20260806T220027579532Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/Raw-CP/Raw-CP-canonical-dt0.005-20260806T220027579532Z',
  'dt': 0.005,
  'calibration_hash': '86f14647ba4dcff44452cac2499c01ea',
  'fee_calibration_hash': '088ecf363630e7e9c4899a3eb827d2f6',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'Raw-CP, tamed',
  'status': 'complete',
  'run_id': 'Raw-CP-tamed-dt0.005-20260806T220049686722Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/Raw-CP/Raw-CP-tamed-dt0.005-20260806T220049686722Z',
  'dt': 0.005,
  'calibration_hash': '13caa62190914eb32240522bced0046d',
  'fee_calibration_hash': '088ecf363630e7e9c4899a3eb827d2f6',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## LSC-CP

Compound-Poisson jumps with the full deterministic-quadrature Levy score correction.

In [9]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[LSC-CP, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/LSC-CP/LSC-CP-tamed-dt0.005-20260806T220456649557Z


[{'variant_label': 'LSC-CP, canonical',
  'method': 'LSC-CP',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.005,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.888632421875)],
    'agreement_failures': [],
    'summary': {'n_steps': 2500,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.888632421875,
     'n_effective': 1024,
     'summary_mean': 0.5001484888689944,
     'summary_mean_se': 0.023334698558979104,
     'summary_abs_mean': 0.7485176609434597,
     'summary_abs_mean_se': 0.014584351709687669,
     'summary_median': 0.690624779221328,
     'summary_median_se': 0.1066545791557059,
     'summary_iqr': 1.113527194223632,
     'summary_iqr_se': 0.013222029988917163,
     'energy_mean': 1.8442767603145889,
     'energy_mean_se': 0.07016199174192184,
     'ene

## LSC-CP-RA

`A` is the **iid Monte Carlo bank size of one estimator family**, LSC-CP-RA. `A = 1, 4, 8` are variants of that single family, not three separate methods, and **all of them run from this one cell** and save as separate variants.

The bank holds `A` displacements drawn iid from the full normalised jump law `rho = nu / lambda`, and **the same bank drives both the score and the compound-Poisson increment**.

In [10]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP-RA",
    variants=[
        {"A": 1}, {"A": 4}, {"A": 8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[LSC-CP-RA, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/LSC-CP-RA/LSC-CP-RA-A1-tamed-dt0.005-20260806T220831830574Z


[LSC-CP-RA (A=4), canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA (A=4), tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/LSC-CP-RA/LSC-CP-RA-A4-tamed-dt0.005-20260806T221207234451Z


[LSC-CP-RA (A=8), canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA (A=8), tamed] saved to /home/zheyuanlai/levy-sampling/results/E3_muller_brown/runs/LSC-CP-RA/LSC-CP-RA-A8-tamed-dt0.005-20260806T221544570544Z


[{'variant_label': 'LSC-CP-RA, canonical',
  'method': 'LSC-CP-RA',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.005,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.50699375)],
    'agreement_failures': [{'key': 'energy_iqr',
      'coarse': 0.3378592,
      'fine': 0.60465659,
      'difference': 0.26679739,
      'allowance': 0.20920481}],
    'summary': {'n_steps': 2500,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.50699375,
     'n_effective': 1024,
     'summary_mean': -0.27164207392486617,
     'summary_mean_se': 0.012626831642500008,
     'summary_abs_mean': 0.3803562751352657,
     'summary_abs_mean_se': 0.007940447110204172,
     'summary_median': -0.2813963691579891,
     'summary_median_se': 0.01938131319127623,
     'summary_iqr': 0.493904034512844

## Rebuild the catalog

`catalog.csv` is a **derived index** over the run manifests. It is never written by a worker mid-run, so concurrent runs never contend for it, and it can be rebuilt at any time by rescanning the manifests -- a lost or stale catalog costs nothing. Only runs that verify (manifest present, `COMPLETE` present, hashes matching) are admitted.

In [11]:
from src.catalog import write_catalog

report = write_catalog(experiment.paths.experiment_dir)
print(f"catalog rebuilt: {report['n_runs']} runs, {report['n_rejected']} rejected")

catalog rebuilt: 72 runs, 29 rejected
